# 1. 목적과 비-Chunking 원칙

이 Notebook은 `03_document_json/{INPUT_RUN_ID}`의 법률 구조 Document를 의미 보존형으로
전처리하고 Text/Image Document를 분리한다. 토큰 Chunking과 자동 dedup은 수행하지 않으며,
heading·삭제 조문·개정 정보·모든 판례 관련성 등급을 보존한다.


## 2. `INPUT_RUN_ID`와 경로 설정


In [1]:
from __future__ import annotations

import html
import json
import os
import re
import shutil
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable, Iterator
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".git").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root()
INPUT_RUN_ID = os.getenv("LEGAL_API_INPUT_RUN_ID", "sample_20260718_232000").strip()
if not re.fullmatch(r"(?:sample|full)_\d{8}_\d{6}", INPUT_RUN_ID):
    raise ValueError("LEGAL_API_INPUT_RUN_ID 형식은 sample_YYYYMMDD_HHMMSS 또는 full_YYYYMMDD_HHMMSS여야 합니다.")

DATA_ROOT = PROJECT_ROOT / "data" / "legal_api_v2"
INPUT_DIR = DATA_ROOT / "03_document_json" / INPUT_RUN_ID
PROCESSED_MD_ROOT = DATA_ROOT / "04_processed_md" / INPUT_RUN_ID
PROCESSED_JSON_ROOT = DATA_ROOT / "05_processed_document_json" / INPUT_RUN_ID
TEXT_JSON_DIR = PROCESSED_JSON_ROOT / "text"
IMAGE_JSON_DIR = PROCESSED_JSON_ROOT / "image"
TEXT_MD_DIR = PROCESSED_MD_ROOT / "text"
IMAGE_REVIEW_DIR = PROCESSED_MD_ROOT / "image_review"
JSONL_MAX_BYTES = 50 * 1024 * 1024
MARKDOWN_MAX_DOCUMENTS = 200

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"입력 run 디렉터리가 없습니다: {INPUT_DIR}")
print(f"INPUT_RUN_ID: {INPUT_RUN_ID}")
print(f"입력: {INPUT_DIR.relative_to(PROJECT_ROOT)}")
print(f"출력: {PROCESSED_MD_ROOT.relative_to(PROJECT_ROOT)}, {PROCESSED_JSON_ROOT.relative_to(PROJECT_ROOT)}")


INPUT_RUN_ID: full_20260719_163719
입력: data\legal_api_v2\03_document_json\full_20260719_163719
출력: data\legal_api_v2\04_processed_md\full_20260719_163719, data\legal_api_v2\05_processed_document_json\full_20260719_163719


## 3. 입력 Document inventory·schema 검증


In [2]:
REQUIRED_INPUT_METADATA = {
    "source_type", "source_id", "parent_id", "record_id", "doc_title",
    "source_org", "doc_year", "authority", "stage", "issue", "source_file", "section",
}
FORBIDDEN_CHUNK_FIELDS = {"chunk_id", "chunk_index", "overlap"}


def iter_jsonl(path: Path) -> Iterator[dict[str, Any]]:
    with path.open(encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL 파싱 실패: {path.name}:{line_no}") from exc
            if not isinstance(row, dict):
                raise TypeError(f"Document가 객체가 아닙니다: {path.name}:{line_no}")
            yield row


input_files = sorted(INPUT_DIR.glob("*.jsonl"), key=lambda path: path.name)
if not input_files:
    raise FileNotFoundError(f"입력 JSONL이 없습니다: {INPUT_DIR}")
invalid_names = [path.name for path in input_files if not (path.name == "eflaw.jsonl" or path.name == "expc.jsonl" or path.name.startswith("prec_"))]
if invalid_names:
    raise AssertionError(f"허용되지 않은 입력 파일명: {invalid_names}")

input_records_by_file: dict[str, list[dict[str, Any]]] = {}
inventory: list[dict[str, Any]] = []
input_missing_metadata: list[tuple[str, str, list[str]]] = []
input_forbidden_fields: list[tuple[str, str]] = []
for path in input_files:
    rows = list(iter_jsonl(path))
    input_records_by_file[path.name] = rows
    type_counts = Counter(row.get("metadata", {}).get("source_type", "missing") for row in rows)
    section_counts = Counter(row.get("metadata", {}).get("section", "missing") for row in rows)
    for row in rows:
        metadata = row.get("metadata")
        if not isinstance(row.get("page_content"), str) or not isinstance(metadata, dict):
            raise TypeError(f"Document 계약 위반: {path.name}")
        missing = sorted(REQUIRED_INPUT_METADATA - set(metadata))
        if missing:
            input_missing_metadata.append((path.name, str(metadata.get("record_id", "")), missing))
        if FORBIDDEN_CHUNK_FIELDS.intersection(metadata):
            input_forbidden_fields.append((path.name, str(metadata.get("record_id", ""))))
    inventory.append({"file": path.name, "records": len(rows), "source_types": dict(type_counts), "sections": dict(section_counts)})

assert not input_missing_metadata, input_missing_metadata[:5]
assert not input_forbidden_fields, input_forbidden_fields[:5]
INPUT_RECORD_COUNT = sum(row["records"] for row in inventory)
print(json.dumps({"input_records": INPUT_RECORD_COUNT, "files": inventory}, ensure_ascii=False, indent=2))


{
  "input_records": 8419,
  "files": [
    {
      "file": "eflaw.jsonl",
      "records": 3146,
      "source_types": {
        "statute": 3146
      },
      "sections": {
        "article": 2514,
        "supplement": 343,
        "annex": 45,
        "heading": 244
      }
    },
    {
      "file": "expc.jsonl",
      "records": 962,
      "source_types": {
        "interpretation": 962
      },
      "sections": {
        "question": 321,
        "answer": 320,
        "reason": 321
      }
    },
    {
      "file": "prec_갱신종료.jsonl",
      "records": 1302,
      "source_types": {
        "precedent": 1302
      },
      "sections": {
        "holding": 368,
        "summary": 310,
        "body": 624
      }
    },
    {
      "file": "prec_경매배당.jsonl",
      "records": 585,
      "source_types": {
        "precedent": 585
      },
      "sections": {
        "summary": 168,
        "body": 258,
        "holding": 159
      }
    },
    {
      "file": "prec_보증금권리.jsonl",
    

## 4. 의미 보존형 텍스트 전처리 함수


In [3]:
TABLE_VERTICAL_RE = re.compile(r"[│┃║]")
TABLE_HORIZONTAL_RE = re.compile(r"[─━═┄┅┈┉╌╍]+")
TABLE_JUNCTION_RE = re.compile(r"[┌┐└┘├┤┬┴┼┏┓┗┛┣┫┳┻╋╔╗╚╝╠╣╦╩╬]")
TABLE_LAYOUT_ONLY_RE = re.compile(r"^[\s│┃║─━═┄┅┈┉╌╍┌┐└┘├┤┬┴┼┏┓┗┛┣┫┳┻╋╔╗╚╝╠╣╦╩╬|+_-]+$")


def flatten_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return "\n".join(part for item in value if (part := flatten_text(item)))
    if isinstance(value, dict):
        if value.get("content") not in (None, ""):
            return flatten_text(value["content"])
        return "\n".join(part for item in value.values() if (part := flatten_text(item)))
    return str(value)


def normalize_meaning_preserving_lines(value: Any, *, table: bool = False) -> str:
    text = unicodedata.normalize("NFC", html.unescape(flatten_text(value)))
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00a0", " ")
    lines: list[str] = []
    for raw_line in text.split("\n"):
        line = raw_line
        if table:
            if TABLE_LAYOUT_ONLY_RE.fullmatch(line):
                continue
            line = TABLE_VERTICAL_RE.sub(" | ", line)
            line = TABLE_HORIZONTAL_RE.sub(" ", line)
            line = TABLE_JUNCTION_RE.sub(" ", line)
            line = re.sub(r"\s*\|\s*", " | ", line)
            line = re.sub(r"(?:\s*\|\s*){2,}", " | ", line)
        line = re.sub(r"[ \t]+", " ", line).strip()
        if table:
            line = line.strip(" |").strip()
        lines.append(line)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def unique_values(*values: Any) -> list[Any]:
    result: list[Any] = []
    for value in values:
        items = value if isinstance(value, list) else ([] if value in (None, "") else [value])
        for item in items:
            if item not in (None, "") and item not in result:
                result.append(item)
    return result


## 5. HTML allowlist와 이미지 추출 함수


In [4]:
IMAGE_TAG_RE = re.compile(r"<img\b[^>]*>", re.IGNORECASE)
IMAGE_ATTR_RE = re.compile(r"([:\w-]+)\s*=\s*([\"'])(.*?)\2", re.IGNORECASE | re.DOTALL)
MARKDOWN_IMAGE_RE = re.compile(r"!\[([^\]]*)\]\(([^)\s]+)(?:\s+[\"'][^\"']*[\"'])?\)")
BLOCK_TAG_RE = re.compile(r"</?(?:p|div|li|tr|table|thead|tbody|tfoot|ul|ol|h[1-6])\b[^>]*>", re.IGNORECASE)
BREAK_TAG_RE = re.compile(r"<br\s*/?>", re.IGNORECASE)
CELL_TAG_RE = re.compile(r"</?(?:td|th)\b[^>]*>", re.IGNORECASE)
INLINE_TAG_RE = re.compile(r"</?(?:span|font|b|strong|em|i|u|small|sup|sub|a)\b[^>]*>", re.IGNORECASE)


def sanitize_url(value: Any) -> str:
    raw = html.unescape(str(value or "").strip())
    if not raw:
        return ""
    parts = urlsplit(raw)
    query = [(key, val) for key, val in parse_qsl(parts.query, keep_blank_values=True) if key.lower() != "oc"]
    return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(query, doseq=True), parts.fragment))


def image_id_from_url(url: str) -> str:
    for key, value in parse_qsl(urlsplit(url).query, keep_blank_values=True):
        if key.lower() in {"flseq", "id", "imageid", "imgid"} and value:
            return value
    return ""


def extract_inline_images(value: Any, source_section: str) -> list[dict[str, Any]]:
    raw = flatten_text(value)
    images: list[dict[str, Any]] = []
    for tag in IMAGE_TAG_RE.findall(raw):
        attrs = {name.lower(): val for name, _, val in IMAGE_ATTR_RE.findall(tag)}
        url = sanitize_url(attrs.get("src", ""))
        image_id = attrs.get("id", "") or image_id_from_url(url)
        images.append({
            "image_id": image_id, "image_url": url, "alt": attrs.get("alt", ""),
            "title": attrs.get("title", ""), "description": "", "source_section": source_section,
        })
    for alt, raw_url in MARKDOWN_IMAGE_RE.findall(raw):
        url = sanitize_url(raw_url)
        images.append({
            "image_id": image_id_from_url(url), "image_url": url, "alt": alt,
            "title": "", "description": "", "source_section": source_section,
        })
    return images


def normalize_image_references(metadata_images: Any, inline_images: list[dict[str, Any]]) -> list[dict[str, Any]]:
    combined: list[dict[str, Any]] = []
    seen: set[tuple[str, str, str]] = set()
    raw_items = metadata_images if isinstance(metadata_images, list) else ([] if metadata_images in (None, "") else [metadata_images])
    for raw in [*raw_items, *inline_images]:
        if not isinstance(raw, dict):
            continue
        url = sanitize_url(raw.get("image_url") or raw.get("url") or raw.get("src"))
        image_id = str(raw.get("image_id") or raw.get("id") or image_id_from_url(url) or "").strip()
        source_section = str(raw.get("source_section") or "page_content").strip()
        marker = (image_id, url, source_section)
        if marker in seen:
            continue
        seen.add(marker)
        combined.append({
            "image_id": image_id,
            "image_url": url,
            "alt": flatten_text(raw.get("alt")),
            "title": flatten_text(raw.get("title")),
            "description": flatten_text(raw.get("description")),
            "source_section": source_section,
        })
    for order, item in enumerate(combined, 1):
        item["image_order"] = order
    return combined


def preprocess_text(value: Any, *, table: bool = False) -> str:
    text = flatten_text(value)
    text = IMAGE_TAG_RE.sub("", text)
    text = MARKDOWN_IMAGE_RE.sub("", text)
    text = BREAK_TAG_RE.sub("\n", text)
    text = CELL_TAG_RE.sub(" | ", text)
    text = BLOCK_TAG_RE.sub("\n", text)
    text = INLINE_TAG_RE.sub("", text)
    # Allowlist 밖의 꺾쇠 텍스트(예: <신설 ...>, <General>)는 손대지 않는다.
    return normalize_meaning_preserving_lines(text, table=table)


def prepare_text_document(record: dict[str, Any]) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    metadata = dict(record["metadata"])
    inline_images = extract_inline_images(record["page_content"], str(metadata.get("section", "page_content")))
    images = normalize_image_references(metadata.get("images", []), inline_images)
    is_table = metadata.get("section") == "annex"
    content = preprocess_text(record["page_content"], table=is_table)
    metadata.update({"content_type": "text", "has_image": bool(images), "image_count": len(images), "images": images})
    return {"page_content": content, "metadata": metadata}, images


## 6. 법령 전처리


In [5]:
statute_prepared: list[tuple[str, dict[str, Any], list[dict[str, Any]]]] = []
for filename, records in input_records_by_file.items():
    for record in records:
        if record["metadata"].get("source_type") != "statute":
            continue
        prepared, image_refs = prepare_text_document(record)
        statute_prepared.append((filename, prepared, image_refs))

statute_sections = Counter(item[1]["metadata"].get("section") for item in statute_prepared)
print(json.dumps({"statute_text_records": len(statute_prepared), "sections": dict(statute_sections)}, ensure_ascii=False))


{"statute_text_records": 3146, "sections": {"article": 2514, "supplement": 343, "annex": 45, "heading": 244}}


## 7. 판례 관련성 평가 및 섹션 전처리


In [6]:
RENTAL_ANCHORS = [
    "임대차", "임대인", "임차인", "임차권", "보증금", "차임",
    "전세", "월세", "중개", "대항력", "우선변제",
]
ANCHOR_PATTERNS = {"전세": re.compile(r"전세(?!계)")}
TOPIC_TERMS = {
    "보증금권리": ["보증금", "대항력", "우선변제", "최우선변제", "확정일자", "임차권등기", "전입신고"],
    "갱신종료": ["갱신", "해지", "차임", "연체", "계약종료"],
    "전세사기": ["전세사기", "가장임대차", "무권대리", "이중계약", "사해행위", "명의신탁"],
    "경매배당": ["경매", "배당", "인도명령", "건물명도", "우선변제"],
    "수선원상회복": ["수선", "원상회복", "누수", "하자", "통상의 손모"],
    "중개": ["공인중개사", "중개대상물", "확인설명", "중개보수", "중개업자"],
}


def contains_anchor(text: str, anchor: str) -> bool:
    pattern = ANCHOR_PATTERNS.get(anchor)
    return bool(pattern.search(text)) if pattern else anchor in text


def classify_precedent(
    title: str, holdings: str, summary: str, references: str,
    body: str, categories: list[str], court: str,
) -> tuple[str, int, list[str], list[str], list[str], list[str], str | None]:
    high_sections = {"사건명": title, "판시사항": holdings, "판결요지": summary, "참조조문": references}
    all_text = "\n".join([*high_sections.values(), body])
    topic_terms = list(dict.fromkeys(term for category in categories for term in TOPIC_TERMS.get(category, [])))
    matched_anchors = [term for term in RENTAL_ANCHORS if contains_anchor(all_text, term)]
    matched_anchor_sections = [
        section for section, text in high_sections.items()
        if any(contains_anchor(text, term) for term in RENTAL_ANCHORS)
    ]
    matched_terms = [term for term in topic_terms if term in all_text]
    matched_sections = [
        section for section, text in high_sections.items()
        if any(term in text for term in topic_terms)
    ]

    score = len(matched_anchors)
    for section, weight in {"사건명": 3, "판시사항": 5, "판결요지": 4, "참조조문": 4}.items():
        if any(term in high_sections[section] for term in topic_terms):
            score += weight
    if any(term in body for term in topic_terms):
        score += 1
    if "대법원" in court:
        score += 1

    if matched_anchor_sections and matched_sections and score >= 5:
        level, reason = "relevant", None
    elif matched_anchor_sections or (matched_anchors and matched_terms):
        level, reason = "candidate", "관련 문맥은 있으나 핵심 쟁점 여부가 불명확함"
    else:
        level, reason = "excluded", "임대차 핵심어와 카테고리 쟁점을 확인하지 못함"
    return level, score, matched_anchors, matched_anchor_sections, matched_terms, matched_sections, reason


precedent_prepared: list[tuple[str, dict[str, Any], list[dict[str, Any]]]] = []
precedent_groups: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
for filename, records in input_records_by_file.items():
    for record in records:
        if record["metadata"].get("source_type") != "precedent":
            continue
        prepared, image_refs = prepare_text_document(record)
        precedent_prepared.append((filename, prepared, image_refs))
        precedent_groups[(filename, str(prepared["metadata"]["source_id"]))].append(prepared)

precedent_classification: dict[tuple[str, str], str] = {}
for (filename, source_id), records in precedent_groups.items():
    by_section = {record["metadata"].get("section"): record["page_content"] for record in records}
    first = records[0]["metadata"]
    fallback_category = Path(filename).stem.removeprefix("prec_")
    categories = [str(item) for item in unique_values(first.get("collection_categories"), fallback_category)]
    result = classify_precedent(
        preprocess_text(first.get("doc_title", "")), by_section.get("holding", ""), by_section.get("summary", ""),
        preprocess_text(first.get("reference_laws", "")), by_section.get("body", ""), categories,
        preprocess_text(first.get("court") or first.get("source_org", "")),
    )
    level, score, anchors, anchor_sections, terms, matched_sections, reason = result
    precedent_classification[(filename, source_id)] = level
    for record in records:
        metadata = record["metadata"]
        metadata.update({
            "relevance_level": level, "relevance_score": score,
            "matched_anchors": anchors, "matched_anchor_sections": anchor_sections,
            "matched_terms": terms, "matched_sections": matched_sections,
        })
        if reason is None:
            metadata.pop("exclusion_reason", None)
        else:
            metadata["exclusion_reason"] = reason

relevance_distribution = Counter(precedent_classification.values())
print(json.dumps({"precedent_cases": len(precedent_groups), "precedent_text_records": len(precedent_prepared), "relevance": dict(relevance_distribution)}, ensure_ascii=False))


{"precedent_cases": 2127, "precedent_text_records": 4311, "relevance": {"excluded": 304, "relevant": 653, "candidate": 1170}}


## 8. 해석례 전처리


In [7]:
interpretation_prepared: list[tuple[str, dict[str, Any], list[dict[str, Any]]]] = []
for filename, records in input_records_by_file.items():
    for record in records:
        if record["metadata"].get("source_type") != "interpretation":
            continue
        prepared, image_refs = prepare_text_document(record)
        interpretation_prepared.append((filename, prepared, image_refs))

interpretation_sections = Counter(item[1]["metadata"].get("section") for item in interpretation_prepared)
print(json.dumps({"interpretation_text_records": len(interpretation_prepared), "sections": dict(interpretation_sections)}, ensure_ascii=False))


{"interpretation_text_records": 962, "sections": {"question": 321, "answer": 320, "reason": 321}}


## 9. Text/Image Document 분리


In [8]:
def image_description(image: dict[str, Any]) -> str:
    for key in ("description", "alt", "title"):
        if image.get(key):
            return preprocess_text(image[key])
    return ""


def image_document(parent: dict[str, Any], image: dict[str, Any]) -> dict[str, Any]:
    parent_meta = parent["metadata"]
    order = int(image["image_order"])
    content = image_description(image)
    metadata = {key: value for key, value in parent_meta.items() if key != "images"}
    metadata.update({
        "record_id": f'{parent_meta["record_id"]}:image:{order}',
        "parent_record_id": parent_meta["record_id"],
        "content_type": "image",
        "image_id": image.get("image_id", ""),
        "image_url": image.get("image_url", ""),
        "image_order": order,
        "source_section": image.get("source_section") or parent_meta.get("section", ""),
        "ocr_status": "not_attempted",
        "text_extraction_status": "metadata_text" if content else "image_only",
        "has_image": True,
        "image_count": 1,
    })
    return {"page_content": content, "metadata": metadata}


all_prepared = [*statute_prepared, *precedent_prepared, *interpretation_prepared]
text_records_by_file: dict[str, list[dict[str, Any]]] = defaultdict(list)
image_records: list[dict[str, Any]] = []
for filename, text_document, image_refs in all_prepared:
    text_records_by_file[filename].append(text_document)
    image_records.extend(image_document(text_document, image) for image in image_refs)


def reset_output_directory(path: Path, allowed_parent: Path) -> None:
    resolved = path.resolve()
    if resolved.parent != allowed_parent.resolve():
        raise RuntimeError(f"출력 삭제 안전성 검사 실패: {resolved}")
    if resolved.exists():
        shutil.rmtree(resolved)
    resolved.mkdir(parents=True, exist_ok=True)


reset_output_directory(PROCESSED_MD_ROOT, DATA_ROOT / "04_processed_md")
reset_output_directory(PROCESSED_JSON_ROOT, DATA_ROOT / "05_processed_document_json")
TEXT_JSON_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_JSON_DIR.mkdir(parents=True, exist_ok=True)


def jsonl_parts(records: list[dict[str, Any]]) -> list[list[bytes]]:
    parts: list[list[bytes]] = [[]]
    size = 0
    for record in records:
        encoded = (json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n").encode("utf-8")
        if parts[-1] and size + len(encoded) > JSONL_MAX_BYTES:
            parts.append([])
            size = 0
        parts[-1].append(encoded)
        size += len(encoded)
    return parts


def write_jsonl_limited(path: Path, records: list[dict[str, Any]]) -> list[Path]:
    path.parent.mkdir(parents=True, exist_ok=True)
    parts = jsonl_parts(records)
    if len(parts) == 1:
        paths = [path]
    else:
        paths = [path.with_name(f"{path.stem}_part{index:02d}{path.suffix}") for index in range(1, len(parts) + 1)]
    for output_path, lines in zip(paths, parts):
        output_path.write_bytes(b"".join(lines))
    return paths


json_output_paths: list[Path] = []
for filename in sorted(text_records_by_file):
    json_output_paths.extend(write_jsonl_limited(TEXT_JSON_DIR / filename, text_records_by_file[filename]))
json_output_paths.extend(write_jsonl_limited(IMAGE_JSON_DIR / "images.jsonl", image_records))
print(json.dumps({"text_records": sum(map(len, text_records_by_file.values())), "image_records": len(image_records)}, ensure_ascii=False))


{"text_records": 8419, "image_records": 119}


## 10. 원문 문서별 `text` Markdown 생성


In [9]:
SECTION_TITLES = {
    "heading": "구조 제목", "article": "조문", "supplement": "부칙", "annex": "별표",
    "holding": "판시사항", "summary": "판결요지", "body": "판례내용",
    "question": "질의요지", "answer": "회답", "reason": "이유",
}


def safe_name(value: Any) -> str:
    name = re.sub(r'[<>:"/\\|?*]+', "_", str(value or "기타")).strip().rstrip(".")
    return name or "기타"


def markdown_group_key(record: dict[str, Any]) -> tuple[str, str]:
    metadata = record["metadata"]
    source_type = metadata["source_type"]
    if source_type == "statute":
        return "eflaw", safe_name(metadata.get("law_name") or metadata.get("doc_title"))
    categories = unique_values(metadata.get("collection_categories"), metadata.get("issue"))
    category = safe_name(categories[0] if categories else "기타")
    return ("prec", category) if source_type == "precedent" else ("expc", category)


def text_document_blocks(records: list[dict[str, Any]]) -> list[str]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        grouped[str(record["metadata"]["source_id"])].append(record)
    blocks: list[str] = []
    for source_records in grouped.values():
        metadata = source_records[0]["metadata"]
        lines = [f'# {metadata.get("doc_title") or metadata.get("source_id")}', f'<!-- source_id: {metadata.get("source_id")} -->', ""]
        for record in source_records:
            section = record["metadata"].get("section", "")
            title = record["metadata"].get("section_title") or SECTION_TITLES.get(section, section)
            if section == "heading":
                title = record["page_content"]
            lines.extend([f"## {title}", "", record["page_content"], ""])
        blocks.append("\n".join(lines).strip() + "\n")
    return blocks


def write_markdown_limited(directory: Path, stem: str, blocks: list[str]) -> list[Path]:
    directory.mkdir(parents=True, exist_ok=True)
    parts: list[list[str]] = [[]]
    size = 0
    for block in blocks:
        encoded_size = len((block + "\n").encode("utf-8"))
        if parts[-1] and (len(parts[-1]) >= MARKDOWN_MAX_DOCUMENTS or size + encoded_size > JSONL_MAX_BYTES):
            parts.append([])
            size = 0
        parts[-1].append(block)
        size += encoded_size
    paths = [directory / f"{stem}.md"] if len(parts) == 1 else [directory / f"{stem}_part{index:02d}.md" for index in range(1, len(parts) + 1)]
    for path, part in zip(paths, parts):
        path.write_text("\n".join(part).rstrip() + "\n", encoding="utf-8", newline="\n")
    return paths


text_markdown_groups: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
for records in text_records_by_file.values():
    for record in records:
        text_markdown_groups[markdown_group_key(record)].append(record)

text_markdown_paths: list[Path] = []
for (target, stem), records in sorted(text_markdown_groups.items()):
    text_markdown_paths.extend(write_markdown_limited(TEXT_MD_DIR / target, stem, text_document_blocks(records)))
print(f"text Markdown 파일: {len(text_markdown_paths)}")


text Markdown 파일: 32


## 11. 문서별 `image_review` Markdown 생성


In [10]:
image_review_groups: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
for record in image_records:
    image_review_groups[markdown_group_key(record)].append(record)


def image_review_blocks(records: list[dict[str, Any]]) -> list[str]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        grouped[str(record["metadata"]["parent_record_id"])].append(record)
    blocks: list[str] = []
    for parent_record_id, source_records in grouped.items():
        first = source_records[0]["metadata"]
        lines = [f'# {first.get("doc_title") or first.get("source_id")}', f'<!-- parent_record_id: {parent_record_id} -->', ""]
        for record in source_records:
            metadata = record["metadata"]
            lines.extend([
                f'## 이미지 {metadata["image_order"]}', "",
                f'- image_id: {metadata.get("image_id", "")}',
                f'- source_section: {metadata.get("source_section", "")}',
                f'- ocr_status: {metadata.get("ocr_status", "")}',
            ])
            if metadata.get("image_url"):
                lines.append(f'- 이미지 URL: [원본 참조]({metadata["image_url"]})')
            if record["page_content"]:
                lines.extend(["", record["page_content"]])
            lines.append("")
        blocks.append("\n".join(lines).strip() + "\n")
    return blocks


image_markdown_paths: list[Path] = []
for (target, stem), records in sorted(image_review_groups.items()):
    image_markdown_paths.extend(write_markdown_limited(IMAGE_REVIEW_DIR / target, stem, image_review_blocks(records)))
print(f"image_review Markdown 파일: {len(image_markdown_paths)}")


image_review Markdown 파일: 8


## 12. metadata·ID·빈 본문 예외·보안 문자열·정보 손실 검증


In [11]:
def all_output_records() -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    text_rows = [row for path in sorted(TEXT_JSON_DIR.glob("*.jsonl")) for row in iter_jsonl(path)]
    image_rows = [row for path in sorted(IMAGE_JSON_DIR.glob("*.jsonl")) for row in iter_jsonl(path)]
    return text_rows, image_rows


output_text_records, output_image_records = all_output_records()
output_text_by_id = {record["metadata"].get("record_id"): record for record in output_text_records}
input_records = [record for records in input_records_by_file.values() for record in records]
input_by_id = {record["metadata"].get("record_id"): record for record in input_records}
all_records = [*output_text_records, *output_image_records]
all_ids = [record["metadata"].get("record_id") for record in all_records]

heading_ids = {record["metadata"]["record_id"] for record in input_records if record["metadata"].get("section") == "heading"}
deleted_ids = {record["metadata"]["record_id"] for record in input_records if record["metadata"].get("is_deleted") is True}
revision_ids = {record["metadata"]["record_id"] for record in input_records if record["metadata"].get("revision_notes") not in (None, "", [])}


def structure_paths_are_consistent() -> bool:
    hierarchy = {"part": "", "chapter": "", "division": "", "subdivision": ""}
    levels = list(hierarchy)
    current_source_id: str | None = None
    for record in output_text_records:
        metadata = record["metadata"]
        if metadata.get("source_type") != "statute":
            continue
        source_id = str(metadata.get("source_id", ""))
        if source_id != current_source_id:
            hierarchy = {level: "" for level in levels}
            current_source_id = source_id
        if metadata.get("section") == "heading":
            changed = next((level for level in levels if metadata.get(level) and metadata.get(level) != hierarchy[level]), None)
            if changed:
                position = levels.index(changed)
                hierarchy[changed] = metadata[changed]
                for lower in levels[position + 1:]:
                    hierarchy[lower] = ""
            expected = [value for value in hierarchy.values() if value]
            if metadata.get("structure_path", []) != expected:
                return False
        elif metadata.get("section") == "article":
            expected = [value for value in hierarchy.values() if value]
            if metadata.get("structure_path", []) != expected:
                return False
    return True


LOGICAL_MARKER_RE = re.compile(r"제\s*\d+조(?:의\d+)?|[①-⑳]|^\s*\d+\.|^\s*[가-하]\.", re.MULTILINE)


def logical_order_preserved() -> bool:
    for record_id, source in input_by_id.items():
        if source["metadata"].get("section") != "article":
            continue
        target = output_text_by_id.get(record_id)
        if target is None:
            return False
        before = LOGICAL_MARKER_RE.findall(html.unescape(source["page_content"]))
        after = LOGICAL_MARKER_RE.findall(target["page_content"])
        if before != after:
            return False
    return True


PROTECTED_ANGLE_RE = re.compile(r"<\s*(?:(?:일부|전문)?개정|신설|삭제)[^>]*>|<General>", re.IGNORECASE)
def normalize_angle_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


protected_before = Counter(normalize_angle_text(match.group(0)) for record in input_records for match in PROTECTED_ANGLE_RE.finditer(record["page_content"]))
protected_after = Counter(normalize_angle_text(match.group(0)) for record in output_text_records for match in PROTECTED_ANGLE_RE.finditer(record["page_content"]))
forbidden_tags_after = [
    record["metadata"].get("record_id") for record in output_text_records
    if IMAGE_TAG_RE.search(record["page_content"]) or BREAK_TAG_RE.search(record["page_content"])
    or BLOCK_TAG_RE.search(record["page_content"]) or CELL_TAG_RE.search(record["page_content"])
    or INLINE_TAG_RE.search(record["page_content"])
]

precedent_input_ids = {record["metadata"]["source_id"] for record in input_records if record["metadata"].get("source_type") == "precedent"}
precedent_output_ids = {record["metadata"]["source_id"] for record in output_text_records if record["metadata"].get("source_type") == "precedent"}
relevance_complete = all(
    {"relevance_level", "relevance_score", "matched_anchors", "matched_terms"}.issubset(record["metadata"])
    and record["metadata"]["relevance_level"] in {"relevant", "candidate", "excluded"}
    and (record["metadata"]["relevance_level"] == "relevant" or bool(record["metadata"].get("exclusion_reason")))
    for record in output_text_records if record["metadata"].get("source_type") == "precedent"
)

text_ids = set(output_text_by_id)
parent_integrity = all(
    record["metadata"].get("parent_record_id") in text_ids
    and output_text_by_id[record["metadata"]["parent_record_id"]]["metadata"].get("has_image") is True
    for record in output_image_records
)
empty_content_policy = all(
    bool(record["page_content"].strip()) for record in output_text_records
) and all(
    bool(record["page_content"].strip()) or record["metadata"].get("text_extraction_status") == "image_only"
    for record in output_image_records
)
forbidden_chunk_absent = all(not FORBIDDEN_CHUNK_FIELDS.intersection(record["metadata"]) for record in all_records)

output_artifacts = [path for root in (PROCESSED_MD_ROOT, PROCESSED_JSON_ROOT) for path in root.rglob("*") if path.is_file()]
query_assignment = ("o" + "c=")
oc_query_assignment_absent = all(query_assignment not in path.read_text(encoding="utf-8", errors="ignore").lower() for path in output_artifacts)

checks = {
    "input_inventory_schema": not input_missing_metadata and not input_forbidden_fields,
    "heading_preservation": heading_ids.issubset(output_text_by_id),
    "structure_path_consistency": structure_paths_are_consistent(),
    "article_unit_order_preservation": logical_order_preserved(),
    "deleted_article_preservation": deleted_ids.issubset(output_text_by_id) and all(output_text_by_id[item]["metadata"].get("is_deleted") is True for item in deleted_ids),
    "revision_information_preservation": revision_ids.issubset(output_text_by_id) and all(output_text_by_id[item]["metadata"].get("revision_notes") == input_by_id[item]["metadata"].get("revision_notes") for item in revision_ids),
    "low_relevance_precedent_preservation": precedent_input_ids == precedent_output_ids,
    "relevance_metadata_complete": relevance_complete,
    "legal_angle_text_preservation": protected_before == protected_after,
    "html_allowlist_handled": not forbidden_tags_after,
    "no_automatic_dedup": len(output_text_records) == len(input_records) and set(input_by_id) == set(output_text_by_id),
    "metadata_and_record_id_integrity": all(all_ids) and len(all_ids) == len(set(all_ids)),
    "text_image_parent_integrity": parent_integrity,
    "image_count_integrity": len(output_image_records) == sum(int(record["metadata"].get("image_count", 0)) for record in output_text_records),
    "empty_content_policy": empty_content_policy,
    "forbidden_chunk_metadata_absent": forbidden_chunk_absent,
    "oc_query_assignment_absent": oc_query_assignment_absent,
    "record_count_conservation": len(output_text_records) == INPUT_RECORD_COUNT,
}

input_type_counts = Counter(record["metadata"]["source_type"] for record in input_records)
output_text_type_counts = Counter(record["metadata"]["source_type"] for record in output_text_records)
output_image_type_counts = Counter(record["metadata"]["source_type"] for record in output_image_records)
validation_result = {
    "run_id": INPUT_RUN_ID,
    "passed": all(checks.values()),
    "checks": checks,
    "record_counts": {
        "input": dict(input_type_counts),
        "output_text": dict(output_text_type_counts),
        "output_image": dict(output_image_type_counts),
        "input_total": len(input_records),
        "output_text_total": len(output_text_records),
        "output_image_total": len(output_image_records),
    },
    "precedent_relevance_distribution": {level: relevance_distribution.get(level, 0) for level in ("relevant", "candidate", "excluded")},
}
validation_path = PROCESSED_MD_ROOT / "preprocess_validation.json"
validation_path.write_text(json.dumps(validation_result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8", newline="\n")
print(json.dumps(validation_result, ensure_ascii=False, indent=2))
assert validation_result["passed"], {key: value for key, value in checks.items() if not value}


{
  "run_id": "full_20260719_163719",
  "passed": true,
  "checks": {
    "input_inventory_schema": true,
    "heading_preservation": true,
    "structure_path_consistency": true,
    "article_unit_order_preservation": true,
    "deleted_article_preservation": true,
    "revision_information_preservation": true,
    "low_relevance_precedent_preservation": true,
    "relevance_metadata_complete": true,
    "legal_angle_text_preservation": true,
    "html_allowlist_handled": true,
    "no_automatic_dedup": true,
    "metadata_and_record_id_integrity": true,
    "text_image_parent_integrity": true,
    "image_count_integrity": true,
    "empty_content_policy": true,
    "forbidden_chunk_metadata_absent": true,
    "oc_query_assignment_absent": true,
    "record_count_conservation": true
  },
  "record_counts": {
    "input": {
      "statute": 3146,
      "interpretation": 962,
      "precedent": 4311
    },
    "output_text": {
      "statute": 3146,
      "interpretation": 962,
      "p

## 13. 유형별 샘플 출력과 완료 보고


In [12]:
generated_files = sorted(
    [PROJECT_ROOT / "notebooks" / "legal_api_v2" / "02_preprocess_legal_api_v2.ipynb"]
    + [path for root in (PROCESSED_MD_ROOT, PROCESSED_JSON_ROOT) for path in root.rglob("*") if path.is_file()],
    key=lambda path: str(path),
)
passed_checks = [name for name, passed in checks.items() if passed]
failed_checks = [name for name, passed in checks.items() if not passed]
samples = {}
for source_type in ("statute", "precedent", "interpretation"):
    sample = next(record for record in output_text_records if record["metadata"]["source_type"] == source_type)
    samples[source_type] = {
        "record_id": sample["metadata"]["record_id"],
        "section": sample["metadata"].get("section"),
        "content_length": len(sample["page_content"]),
    }

print("=== Gate 4 전처리 완료 보고 ===")
print("생성 파일 목록")
for path in generated_files:
    print(f"- {path.relative_to(PROJECT_ROOT).as_posix()}")
print("유형별 입력/출력 레코드 수")
print(json.dumps(validation_result["record_counts"], ensure_ascii=False, indent=2))
print("판례 관련성 등급 분포 (고유 판례 기준)")
print(json.dumps(validation_result["precedent_relevance_distribution"], ensure_ascii=False, indent=2))
print("검증 셀 결과 요약")
print(json.dumps({"passed": validation_result["passed"], "passed_items": passed_checks, "failed_items": failed_checks}, ensure_ascii=False, indent=2))
print("유형별 샘플")
print(json.dumps(samples, ensure_ascii=False, indent=2))


=== Gate 4 전처리 완료 보고 ===
생성 파일 목록
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/eflaw/공인중개사법 시행규칙.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/eflaw/부동산등기규칙.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/eflaw/주택임대차보호법 시행령.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/prec/갱신종료.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/prec/경매배당.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/prec/보증금권리.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/prec/전세사기.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/image_review/prec/중개.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/preprocess_validation.json
- data/legal_api_v2/04_processed_md/full_20260719_163719/text/eflaw/공인중개사법 시행규칙.md
- data/legal_api_v2/04_processed_md/full_20260719_163719/text/eflaw/공인중개사법.md
- data/legal_api_v2/04_processed_md/f